# Cyberbullying Detection on Social Media Using NLP and Random Forest

**College ML/NLP Project — End-to-End Notebook**

This notebook builds a complete pipeline:
1. Acquire and inspect real public datasets
2. Combine + clean them into one dataset (no fabricated data)
3. Explore the data (EDA) with graphs
4. Preprocess text (NLP cleaning)
5. Extract features with TF-IDF
6. Train and compare 4 ML models (Logistic Regression, Naive Bayes, Linear SVM, Random Forest)
7. Evaluate models (accuracy, precision, recall, F1, confusion matrix)
8. Build a `predict_comment()` function
9. Build a rule-based "respectful rewrite" module for comments flagged as cyberbullying
10. Build a simple interactive UI (ipywidgets)
11. Save/reload the trained system with `joblib`

> **Run this in Google Colab.** Cells are meant to be run top to bottom, in order.
> Each section starts with a short markdown explanation you can reuse for your viva.

## 1. Environment Setup

We install/import everything we need up front: `pandas`/`numpy` for data handling,
`matplotlib`/`seaborn` for graphs, `nltk` for text preprocessing, `scikit-learn` for
ML, `wordcloud` for word clouds, and `joblib` to save the trained model.

**Viva point:** we set a fixed `RANDOM_STATE` everywhere so results are reproducible —
running the notebook twice gives the same split, same metrics, same graphs.

In [ ]:
# Run this once per Colab session
!pip install -q wordcloud kagglehub ipywidgets

import os
import re
import string
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results/graphs', exist_ok=True)

print("Environment ready.")

## 2. Dataset Acquisition

We combine **three genuine, publicly available datasets** instead of one, to get
closer to ~200,000 records. We do **not** fabricate rows — if the real usable total
after cleaning is below 200,000, we report the true number (see Section 3.5).

| # | Dataset | Approx. size | Labels | Source |
|---|---------|-------------|--------|--------|
| 1 | **Cyberbullying Classification** (J. Wang et al., 2020) | ~47,700 tweets | 6-class: `age`, `ethnicity`, `gender`, `religion`, `other_cyberbullying`, `not_cyberbullying` | Kaggle: `andrewmvd/cyberbullying-classification` |
| 2 | **Jigsaw Toxic Comment Classification Challenge** | ~159,600 Wikipedia talk-page comments | multi-label: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, `identity_hate` | Kaggle: `julian3833/jigsaw-toxic-comment-classification-challenge` (data originally released by Jigsaw/Conversation AI under CC0) |
| 3 | **Hate Speech and Offensive Language Dataset** (Davidson et al., 2017) | ~24,800 tweets | 3-class: `hate_speech`, `offensive_language`, `neither` | Kaggle: `mrmorj/hate-speech-and-offensive-language-dataset` |

**License / usage notes (state these in your report):**
- Dataset 1: released for research/educational use on Kaggle; cite Wang, Chen, Huang (2020), *"Detecting Cyberbullying Across Multiple Social Media Platforms"*.
- Dataset 2: released by Jigsaw/Conversation AI under **CC0** (public domain) for the Kaggle competition.
- Dataset 3: released by Davidson et al. (2017), *"Automated Hate Speech Detection and the Problem of Offensive Language"*, for research use — cite the paper if used academically.

**Original label definitions differ across datasets** — this matters for how we map
them to a single binary target in Section 3, and we keep the *original* label in a
`category` column so nothing is silently lost.

### How to download (Colab)
Upload your `kaggle.json` API token (Kaggle account → Settings → Create New API Token),
then run the cell below. If you prefer, you can instead manually download the three
CSVs from Kaggle and upload them to `/content/data/`.

In [ ]:
# --- Option A: Kaggle API (recommended) ---
# 1. Upload kaggle.json when prompted, OR place it at ~/.kaggle/kaggle.json
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token)")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fn in uploaded:
        os.rename(fn, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d andrewmvd/cyberbullying-classification -p data --unzip
!kaggle datasets download -d julian3833/jigsaw-toxic-comment-classification-challenge -p data --unzip
!kaggle datasets download -d mrmorj/hate-speech-and-offensive-language-dataset -p data --unzip

print(os.listdir('data'))

In [ ]:
# --- Option B: manual upload fallback ---
# If Option A fails (e.g. no internet/API access), uncomment and upload the CSVs
# you downloaded manually from the Kaggle pages listed above.
#
# from google.colab import files
# uploaded = files.upload()   # select the CSV(s) you downloaded manually

## 3. Load Each Dataset Separately and Inspect

Per the project brief, we load each dataset on its own first, print its shape and
column names, and identify which column is the text and which is the label —
**before** touching the combination step.

In [ ]:
# ---- Dataset 1: Cyberbullying Classification (Wang et al.) ----
df1_raw = pd.read_csv('data/cyberbullying_tweets.csv')
print("Dataset 1 shape:", df1_raw.shape)
print("Dataset 1 columns:", list(df1_raw.columns))
df1_raw.head()

In [ ]:
# Dataset 1 -> text column: 'tweet_text', label column: 'cyberbullying_type'
print(df1_raw['cyberbullying_type'].value_counts())

In [ ]:
# ---- Dataset 2: Jigsaw Toxic Comment Classification ----
df2_raw = pd.read_csv('data/train.csv')   # Jigsaw's file is usually named train.csv
print("Dataset 2 shape:", df2_raw.shape)
print("Dataset 2 columns:", list(df2_raw.columns))
df2_raw.head()

In [ ]:
# Dataset 2 -> text column: 'comment_text'
# label columns: 'toxic','severe_toxic','obscene','threat','insult','identity_hate' (multi-label, 0/1)
toxic_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print(df2_raw[toxic_cols].sum())

In [ ]:
# ---- Dataset 3: Hate Speech and Offensive Language (Davidson et al.) ----
df3_raw = pd.read_csv('data/labeled_data.csv')
print("Dataset 3 shape:", df3_raw.shape)
print("Dataset 3 columns:", list(df3_raw.columns))
df3_raw.head()

In [ ]:
# Dataset 3 -> text column: 'tweet', label column: 'class'
# class: 0 = hate speech, 1 = offensive language, 2 = neither
print(df3_raw['class'].value_counts())

## 4. Normalize Columns, Map Labels to a Consistent Binary Target, and Combine

We now bring all three datasets to the same schema:

`id | comment | label | category | source`

- **`label`**: `1` = Cyberbullying, `0` = Not Cyberbullying (the binary target we train on)
- **`category`**: the *original* label/class from that dataset, preserved so no
  information is silently thrown away
- **`source`**: which dataset the row came from

### Mapping decisions (state these explicitly in your report — this is exactly the
kind of thing a viva panel asks about):

- **Dataset 1** (Wang et al.): `not_cyberbullying` → 0; every other category
  (`age`, `ethnicity`, `gender`, `religion`, `other_cyberbullying`) → 1.
- **Dataset 2** (Jigsaw): a comment is labelled 1 if **any** of
  `toxic/severe_toxic/obscene/threat/insult/identity_hate` is 1, else 0. This is a
  reasonable proxy for "harassing/abusive" but is **not identical** to "cyberbullying"
  — toxicity and cyberbullying are related but not the same construct. We flag this
  as a limitation in the report (Section 20).
- **Dataset 3** (Davidson et al.): `class` 0 (hate speech) or 1 (offensive language) → 1;
  `class` 2 (neither) → 0.

We are combining datasets with **different original definitions of abuse**, so this
is a deliberate simplification, not a silent one — it is documented, and the
`category` column lets you audit it later.

In [ ]:
def normalize_dataset1(df):
    out = pd.DataFrame()
    out['comment'] = df['tweet_text']
    out['category'] = df['cyberbullying_type']
    out['label'] = (df['cyberbullying_type'] != 'not_cyberbullying').astype(int)
    out['source'] = 'wang_cyberbullying_tweets'
    return out

def normalize_dataset2(df):
    out = pd.DataFrame()
    out['comment'] = df['comment_text']
    any_toxic = df[toxic_cols].sum(axis=1) > 0
    out['label'] = any_toxic.astype(int)
    # keep the most specific positive category for readability, else 'none'
    def pick_category(row):
        for c in toxic_cols:
            if row[c] == 1:
                return c
        return 'none'
    out['category'] = df[toxic_cols].apply(pick_category, axis=1)
    out['source'] = 'jigsaw_toxic_comments'
    return out

def normalize_dataset3(df):
    out = pd.DataFrame()
    out['comment'] = df['tweet']
    label_map = {0: 'hate_speech', 1: 'offensive_language', 2: 'neither'}
    out['category'] = df['class'].map(label_map)
    out['label'] = (df['class'] != 2).astype(int)
    out['source'] = 'davidson_hate_offensive'
    return out

df1 = normalize_dataset1(df1_raw)
df2 = normalize_dataset2(df2_raw)
df3 = normalize_dataset3(df3_raw)

for name, d in [('Dataset 1', df1), ('Dataset 2', df2), ('Dataset 3', df3)]:
    print(f"{name}: {d.shape[0]} rows -> label distribution: {d['label'].value_counts().to_dict()}")

In [ ]:
combined = pd.concat([df1, df2, df3], ignore_index=True)
combined.insert(0, 'id', range(1, len(combined) + 1))
print("Combined (before cleaning):", combined.shape)
combined.head()

## 5. Cleaning the Combined Dataset

Steps, in order:
1. Drop rows with missing/empty comments.
2. Strip whitespace and drop comments that are empty after stripping.
3. Drop exact duplicate `comment` text.
4. Handle **conflicting labels**: the same comment text appearing with both label 0
   and label 1 across datasets (this can happen since the three datasets were
   labelled independently). We drop these ambiguous rows rather than guessing.
5. Reset the `id` column and report the final genuine row count.

We do **not** pad the dataset with synthetic rows to hit 200,000 — whatever number
comes out after honest cleaning is the number we report and use.

In [ ]:
before = len(combined)

# 1-2: missing / empty comments
combined['comment'] = combined['comment'].astype(str).str.strip()
combined = combined[combined['comment'].str.len() > 0]
combined = combined.dropna(subset=['comment', 'label'])

# 3: exact duplicate comment text
combined = combined.drop_duplicates(subset=['comment'])

# 4: conflicting labels for identical text across sources
label_counts_per_comment = combined.groupby('comment')['label'].nunique()
conflicting = label_counts_per_comment[label_counts_per_comment > 1].index
n_conflicting = len(conflicting)
combined = combined[~combined['comment'].isin(conflicting)]

combined = combined.reset_index(drop=True)
combined['id'] = range(1, len(combined) + 1)

after = len(combined)
print(f"Rows before cleaning: {before}")
print(f"Rows removed as duplicate/missing/conflicting: {before - after} (of which {n_conflicting} comments had conflicting labels)")
print(f"FINAL genuine usable records: {after}")

if after < 200000:
    print(f"NOTE: combined usable dataset has {after} genuine records, which is below the 200,000 target. "
          f"This is the true, honest number after cleaning -- no synthetic rows were added.")

In [ ]:
combined = combined[['id', 'comment', 'label', 'category', 'source']]
combined.to_csv('data/cyberbullying_combined.csv', index=False)
print("Saved data/cyberbullying_combined.csv")
combined.sample(5, random_state=RANDOM_STATE)

## 6. Exploratory Data Analysis (EDA)

Before modelling, we look at the shape of the data: class balance, category mix,
where rows came from, and how long comments typically are. This matters because
class imbalance affects which metrics (precision/recall/F1, not just accuracy) we
should trust later.

In [ ]:
print("Rows:", combined.shape[0])
print("Columns:", combined.shape[1])
print()
print("Missing values per column:\n", combined.isnull().sum())
print()
print("Duplicate comments remaining:", combined['comment'].duplicated().sum())
print()
print("Class distribution:\n", combined['label'].value_counts())
print()
n_cb = (combined['label'] == 1).sum()
n_not = (combined['label'] == 0).sum()
print(f"Cyberbullying: {n_cb}  |  Not cyberbullying: {n_not}")
print()
print("Category counts:\n", combined['category'].value_counts())
print()
print("Records per source:\n", combined['source'].value_counts())

In [ ]:
combined['comment_length'] = combined['comment'].str.len()
print("Comment length (characters) -- min/mean/max:")
print(combined['comment_length'].min(), combined['comment_length'].mean(), combined['comment_length'].max())

In [ ]:
# Graph 1: Cyberbullying vs Not-cyberbullying count
plt.figure()
ax = sns.countplot(x='label', data=combined, palette=['#4C72B0', '#C44E52'])
ax.set_xticklabels(['Not Cyberbullying (0)', 'Cyberbullying (1)'])
plt.title('Cyberbullying vs Non-Cyberbullying Comments')
plt.xlabel('Class')
plt.ylabel('Number of Comments')
plt.tight_layout()
plt.savefig('results/graphs/01_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Graph 2: Category distribution
plt.figure(figsize=(10, 6))
order = combined['category'].value_counts().index
sns.countplot(y='category', data=combined, order=order, palette='viridis')
plt.title('Category Distribution (Original Labels)')
plt.xlabel('Number of Comments')
plt.ylabel('Category')
plt.tight_layout()
plt.savefig('results/graphs/02_category_distribution.png', dpi=150)
plt.show()

In [ ]:
# Graph 3: Dataset source distribution
plt.figure()
sns.countplot(x='source', data=combined, palette='Set2')
plt.title('Records per Source Dataset')
plt.xlabel('Source')
plt.ylabel('Number of Comments')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('results/graphs/03_source_distribution.png', dpi=150)
plt.show()

In [ ]:
# Graph 4: Comment length distribution
plt.figure()
sns.histplot(combined['comment_length'], bins=50, kde=True, color='teal')
plt.title('Distribution of Comment Lengths')
plt.xlabel('Comment Length (characters)')
plt.ylabel('Frequency')
plt.xlim(0, combined['comment_length'].quantile(0.99))
plt.tight_layout()
plt.savefig('results/graphs/04_comment_length_distribution.png', dpi=150)
plt.show()

In [ ]:
# Graph 5: Comment length distribution by class
plt.figure()
sns.boxplot(x='label', y='comment_length', data=combined, palette=['#4C72B0', '#C44E52'])
plt.title('Comment Length by Class')
plt.xlabel('Class (0 = Not Cyberbullying, 1 = Cyberbullying)')
plt.ylabel('Comment Length (characters)')
plt.ylim(0, combined['comment_length'].quantile(0.99))
plt.tight_layout()
plt.savefig('results/graphs/05_length_by_class.png', dpi=150)
plt.show()

In [ ]:
# Graph 6: Top 20 most frequent words (rough, pre-cleaning, stopwords removed)
from nltk.corpus import stopwords as sw
stop_set = set(sw.words('english'))

def quick_tokens(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
    return [w for w in text.split() if w not in stop_set and len(w) > 2]

sample_for_wordcount = combined['comment'].sample(min(50000, len(combined)), random_state=RANDOM_STATE)
word_counter = Counter()
for c in sample_for_wordcount:
    word_counter.update(quick_tokens(c))

top20 = word_counter.most_common(20)
words, counts = zip(*top20)

plt.figure(figsize=(10, 6))
sns.barplot(x=list(counts), y=list(words), palette='mako')
plt.title('Top 20 Most Frequent Words (sampled, stopwords removed)')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.savefig('results/graphs/06_top20_words.png', dpi=150)
plt.show()

In [ ]:
# Graph 7 (optional): Word clouds for each class
from wordcloud import WordCloud

cb_text = ' '.join(combined.loc[combined['label'] == 1, 'comment'].sample(
    min(20000, (combined['label'] == 1).sum()), random_state=RANDOM_STATE))
not_cb_text = ' '.join(combined.loc[combined['label'] == 0, 'comment'].sample(
    min(20000, (combined['label'] == 0).sum()), random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
wc1 = WordCloud(width=700, height=500, background_color='white', stopwords=stop_set).generate(cb_text)
axes[0].imshow(wc1, interpolation='bilinear')
axes[0].set_title('Cyberbullying Comments')
axes[0].axis('off')

wc2 = WordCloud(width=700, height=500, background_color='white', stopwords=stop_set).generate(not_cb_text)
axes[1].imshow(wc2, interpolation='bilinear')
axes[1].set_title('Non-Cyberbullying Comments')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('results/graphs/07_wordclouds.png', dpi=150)
plt.show()

## 7. Text Preprocessing Pipeline

We build one function, `clean_text()`, applied consistently everywhere (training,
evaluation, and later at prediction time — this consistency matters a lot for a
viva question on "data leakage" / "train-serve skew").

Steps: lowercase → remove URLs → remove HTML tags → handle `@mentions` → handle
`#hashtags` (keep the word, drop the `#`) → remove punctuation/special characters
→ collapse extra whitespace → tokenize → remove stopwords → lemmatize.

**Why we don't blindly strip everything:** punctuation like repeated `!!!` or
ALL-CAPS can be a weak signal of aggression, and some "stopwords" (e.g. negations
like *not*, *no*) change meaning entirely (*"you are not nice"* vs *"you are nice"*).
So we use NLTK's stopword list but explicitly keep negation words.

In [ ]:
lemmatizer = WordNetLemmatizer()
base_stopwords = set(stopwords.words('english'))
negation_words = {'no', 'not', 'nor', "don't", "didn't", "doesn't", "isn't", "aren't",
                   "wasn't", "weren't", "won't", "can't", "cannot", "couldn't",
                   "shouldn't", "wouldn't", "never"}
custom_stopwords = base_stopwords - negation_words

URL_RE = re.compile(r'https?://\S+|www\.\S+')
HTML_RE = re.compile(r'<.*?>')
MENTION_RE = re.compile(r'@\w+')
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s']")
MULTI_SPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(' ', text)
    text = HTML_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)           # remove @username
    text = text.replace('#', ' ')              # keep hashtag word, drop the symbol
    text = NON_ALPHA_RE.sub(' ', text)         # strip punctuation/special chars
    text = MULTI_SPACE_RE.sub(' ', text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Quick self-test
print(clean_text("Check this out!!! http://example.com @john #SoTrue <b>WOW</b>"))

In [ ]:
combined['clean_comment'] = combined['comment'].apply(clean_text)
combined = combined[combined['clean_comment'].str.len() > 0].reset_index(drop=True)

print("Original -> Cleaned examples:\n")
for orig, clean in combined[['comment', 'clean_comment']].sample(5, random_state=RANDOM_STATE).values:
    print("ORIGINAL:", orig[:120])
    print("CLEANED :", clean[:120])
    print('-' * 80)

## 8. Train/Test Split

80% train / 20% test, **stratified** on `label` so both splits keep the same class
ratio, and a fixed `RANDOM_STATE` for reproducibility. We split on the *cleaned*
text before vectorizing, and fit TF-IDF only on the training set (next section) to
avoid data leakage from the test set into the vocabulary/IDF weights.

In [ ]:
X = combined['clean_comment']
y = combined['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

## 9. Feature Extraction: TF-IDF

**Why TF-IDF?** Term Frequency–Inverse Document Frequency scores a word highly in a
document if it appears often *there* but rarely across the *whole corpus* — so
common filler words are downweighted automatically and distinctive, meaningful
words (including slurs/insults relevant to cyberbullying) stand out. It's fast,
works well with linear/sparse-friendly models, and is a strong, well-understood
baseline for text classification — appropriate for a college project where
interpretability and low compute cost matter.

We include **unigrams + bigrams** (`ngram_range=(1,2)`) so short abusive phrases
("shut up", "kill yourself") are captured, not just single words. We cap vocabulary
size with `max_features` and require a minimum document frequency (`min_df`) to
drop extremely rare tokens/noise.

We **fit the vectorizer only on the training set**, then use it to `transform`
(not `fit_transform`) the test set — this is the correct way to avoid data leakage.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20000,
    min_df=3,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train matrix shape:", X_train_tfidf.shape)
print("TF-IDF test matrix shape :", X_test_tfidf.shape)

## 10. Machine Learning Models

We train and compare **four** models on the same TF-IDF features:

1. **Logistic Regression** — a strong, fast, interpretable linear baseline for text.
2. **Multinomial Naive Bayes** — classic, very fast text-classification baseline,
   assumes conditional independence between word features.
3. **Linear SVM** — often the best linear model for high-dimensional sparse text
   data like TF-IDF; we use `LinearSVC` (wrapped with `CalibratedClassifierCV` so we
   can also get probability estimates for the confidence score later).
4. **Random Forest** — an ensemble of decision trees.

**Note on Random Forest + TF-IDF:** Random Forest is not naturally suited to very
high-dimensional, extremely sparse data the way linear models are — trees built on
thousands of sparse TF-IDF columns can be slow to train and prone to overfitting on
noise. We keep Random Forest as required by the assignment, but we reduce
dimensionality it sees (`max_features` was already capped at 20,000, and we also
cap tree count/depth) so it trains in reasonable time, and we discuss this limitation
explicitly when comparing results in Section 11.

In [ ]:
models = {}

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
lr.fit(X_train_tfidf, y_train)
models['Logistic Regression'] = lr

# 2. Multinomial Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
models['Naive Bayes'] = nb

# 3. Linear SVM (calibrated for probability output)
svm_base = LinearSVC(random_state=RANDOM_STATE, class_weight='balanced', max_iter=5000)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_train_tfidf, y_train)
models['Linear SVM'] = svm

# 4. Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=40, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE, class_weight='balanced'
)
rf.fit(X_train_tfidf, y_train)
models['Random Forest'] = rf

print("All 4 models trained:", list(models.keys()))

## 11. Model Evaluation

For every model we compute Accuracy, Precision, Recall and F1-score (using the
`Cyberbullying` class as the positive class), plus the full classification report
and confusion matrix.

**Why not just accuracy?** In cyberbullying detection, classes can be imbalanced
and the *cost of mistakes is asymmetric*: missing a real cyberbullying comment
(a **false negative**) is more harmful than wrongly flagging a harmless comment
(a **false positive**), because a false negative lets real harassment through
unflagged. So **recall** on the cyberbullying class matters a lot, and F1 balances
recall against precision (too much recall alone -> too many false accusations).
Accuracy alone can look good on an imbalanced dataset while still missing many
real cyberbullying cases.

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1})

    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=['Not Cyberbullying', 'Cyberbullying']))
    print()

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
results_df.to_csv('results/model_results.csv', index=False)
results_df

In [ ]:
# Confusion matrices for all four models
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, model) in zip(axes.flatten(), models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('results/graphs/08_all_confusion_matrices.png', dpi=150)
plt.show()

## 12. Model Comparison

We now compare all four models side by side on each metric.

In [ ]:
print(results_df)

In [ ]:
# Graph: Accuracy comparison
plt.figure()
sns.barplot(x='Model', y='Accuracy', data=results_df, palette='crest')
plt.title('Model Comparison - Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/graphs/09_accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# Graph: Precision comparison
plt.figure()
sns.barplot(x='Model', y='Precision', data=results_df, palette='crest')
plt.title('Model Comparison - Precision')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/graphs/10_precision_comparison.png', dpi=150)
plt.show()

In [ ]:
# Graph: Recall comparison
plt.figure()
sns.barplot(x='Model', y='Recall', data=results_df, palette='crest')
plt.title('Model Comparison - Recall')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/graphs/11_recall_comparison.png', dpi=150)
plt.show()

In [ ]:
# Graph: F1-score comparison
plt.figure()
sns.barplot(x='Model', y='F1 Score', data=results_df, palette='crest')
plt.title('Model Comparison - F1 Score')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/graphs/12_f1_comparison.png', dpi=150)
plt.show()

### Choosing the final model

We pick the model with the **highest F1-score** (not just accuracy), because F1
balances precision and recall, and recall on the cyberbullying class is what
protects users from missed harassment. Run the cell above and read off
`results_df` — in most runs on this kind of combined dataset, **Linear SVM** or
**Logistic Regression** wins on F1, since linear models tend to handle
high-dimensional sparse TF-IDF features better than Random Forest. Write the
*actual* winning model name and its numbers into your report — do not assume it in
advance; let the printed `results_df` decide it for your specific run.

In [ ]:
final_model_name = results_df.iloc[0]['Model']
final_model = models[final_model_name]
print(f"Final selected model: {final_model_name}")
print(results_df.iloc[0])

## 13. Confusion Matrix — Final Model

- **True Positive (TP):** actual cyberbullying, predicted cyberbullying (correctly caught)
- **True Negative (TN):** actual not-cyberbullying, predicted not-cyberbullying (correctly cleared)
- **False Positive (FP):** actual not-cyberbullying, predicted cyberbullying (a false alarm)
- **False Negative (FN):** actual cyberbullying, predicted not-cyberbullying — **a real
  cyberbullying comment that slipped through undetected**

**Why false negatives matter most here:** a false negative means a genuinely
harmful comment reaches its target without being flagged or softened — the direct
harm the whole system exists to prevent. A false positive is annoying (a harmless
comment gets flagged) but far less damaging. This is why we favour recall/F1 over
raw accuracy when selecting the final model.

In [ ]:
y_pred_final = final_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
plt.title(f'Confusion Matrix - {final_model_name} (Final Model)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('results/graphs/13_final_confusion_matrix.png', dpi=150)
plt.show()

print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

## 14. Prediction System — `predict_comment()`

This function takes any new comment, applies the **same** `clean_text()` and TF-IDF
`transform()` used in training (never `fit_transform` here — that would leak test-time
data into the vectorizer), predicts the class, and shows a confidence score.

In [ ]:
def predict_comment(comment, model=None, vectorizer=None, verbose=True):
    model = model or final_model
    vectorizer = vectorizer or tfidf

    cleaned = clean_text(comment)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]

    confidence = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(vec)[0]
        confidence = proba[pred]

    label_text = 'Cyberbullying' if pred == 1 else 'Not Cyberbullying'

    if verbose:
        print(f"Original comment : {comment}")
        print(f"Prediction       : {label_text}")
        if confidence is not None:
            print(f"Confidence       : {confidence:.2%}")

    return {'comment': comment, 'label': int(pred), 'label_text': label_text, 'confidence': confidence}

# Try it on fresh, never-before-seen examples (not hard-coded into the classifier)
predict_comment("You are useless and nobody likes you.")
print()
predict_comment("I really liked your presentation.")

## 15. Respectful Comment Transformation (Rule-Based Baseline)

This is a **separate module** from the classifier — the classifier's only job is
detection; this module only runs *after* a comment is flagged as cyberbullying, and
tries to produce a less harmful rewrite.

**Honesty about the approach:** true meaning-preserving paraphrase-while-removing-abuse
is a hard, open NLP problem (it really needs a fine-tuned generative language model
and human evaluation to do well). For a college project we implement a **transparent,
rule-based baseline**:

1. If the comment matches a known **insult/attack pattern** (you are an idiot / you
   are useless / nobody likes you / kill yourself / shut up, etc.), map it to a
   pre-written respectful alternative that keeps the *topic* of disagreement/feedback
   without the personal attack.
2. If no specific pattern matches, fall back to a generic safe template that
   acknowledges disagreement without hostility.
3. We explicitly do **not** just replace swear words with synonyms — that produces
   nonsense and doesn't remove the harassment; instead we replace the whole harmful
   *sentence pattern* with a respectful sentence expressing a similar (but civil)
   communicative intent (disagreement, feedback, request to stop).

**Limitations to state clearly in your report/viva:**
- It only recognizes patterns it has been given — a genuinely novel insult
  phrasing it hasn't seen will fall through to the generic template, which is
  safe but less specific.
- It cannot guarantee perfect preservation of the original intent — some nuance
  is lost by design, in exchange for guaranteeing the output is non-abusive.
- It is **not** a general-purpose paraphraser; it is a safety-first baseline.

In [ ]:
import re
import hashlib

# --- Category -> (detection patterns, multiple respectful rewrite options) ---
# Grouping by *type* of attack (not one rule per exact phrase) lets many different
# real-world wordings map to a small, reviewable set of categories, while still
# giving several different, category-appropriate outputs instead of one fixed line.

REWRITE_CATEGORIES = [
    {
        "name": "general_insult",
        "patterns": [
            r"\b(you\s*(are|'re)|ur)\s+(an?\s+)?(idiot|stupid|dumb|moron|loser|pathetic|worthless|useless|trash|garbage|clown|joke)\b",
        ],
        "rewrites": [
            "I disagree with your opinion.",
            "I don't think that's a fair point, but let's talk it through.",
            "I see this differently than you do.",
        ],
    },
    {
        "name": "exclusion_worthlessness",
        "patterns": [
            r"\bnobody\s+(likes|wants|cares about|needs)\s+you\b",
            r"\byou\s+(don't|do not)\s+belong\s+here\b",
        ],
        "rewrites": [
            "Let's keep the conversation respectful.",
            "I'd rather we focus on the topic than on each other.",
            "That comment doesn't help the discussion — let's stay on topic.",
        ],
    },
    {
        "name": "self_harm_suggestion",
        "patterns": [
            r"\b(kill\s*yourself|kys|go\s+die|end\s+your\s+life)\b",
        ],
        "rewrites": [
            "I think we should stop this conversation here.",
            "This conversation needs to end — that kind of comment isn't okay.",
        ],
    },
    {
        "name": "silencing_hostility",
        "patterns": [
            r"\bshut\s+up\b",
            r"\bget\s+lost\b",
            r"\bnobody\s+asked\s+you\b",
        ],
        "rewrites": [
            "I'd prefer we discuss this calmly.",
            "Let's slow down and talk this through respectfully.",
        ],
    },
    {
        "name": "appearance_body_shaming",
        "patterns": [
            r"\b(ugly|fat|disgusting|gross)\b",
        ],
        "rewrites": [
            "I don't think that comment adds anything constructive.",
            "Let's keep comments about the topic, not appearance.",
        ],
    },
    {
        "name": "hatred_hostility",
        "patterns": [
            r"\b(hate|despise)\s+you\b",
        ],
        "rewrites": [
            "I strongly disagree with you on this.",
            "We clearly see this very differently.",
        ],
    },
    {
        "name": "threat",
        "patterns": [
            r"\b(i('| a)?ll|i\s+will)\s+(hurt|kill|destroy)\s+you\b",
            r"\byou('re| are)\s+dead\b",
        ],
        "rewrites": [
            "I'm very upset, but I won't threaten you.",
            "This has gone too far — I'm stepping away from this conversation.",
        ],
    },
]

# Fallback pool used only when NO category pattern matches at all.
# Multiple options (not one fixed sentence) chosen deterministically per-comment.
GENERIC_FALLBACKS = [
    "I disagree, but let's keep this discussion respectful.",
    "I don't think that comment is helpful — can we discuss this calmly?",
    "Let's focus on the actual disagreement, not on attacking each other.",
    "I hear that you're frustrated, but let's keep this civil.",
]


def _pick(options, key_text):
    """Deterministic 'pseudo-random' choice so the same input always gives the
    same output (reproducible for grading/demo), but different inputs spread
    across the available options instead of collapsing to one sentence."""
    idx = int(hashlib.md5(key_text.encode()).hexdigest(), 16) % len(options)
    return options[idx]


def respectful_rewrite(comment):
    text = comment.lower()
    for cat in REWRITE_CATEGORIES:
        for pat in cat["patterns"]:
            if re.search(pat, text):
                return _pick(cat["rewrites"], comment)
    # Nothing matched a known category -> still vary the fallback per-comment
    return _pick(GENERIC_FALLBACKS, comment)


# Demo: notice different inputs -> different, category-appropriate outputs
examples = [
    "You are an idiot.",
    "Nobody wants you here.",
    "You are so pathetic and ugly.",
    "Shut up, nobody asked you.",
    "I hate you so much.",
    "This is a completely random comment with no known pattern.",
]
for example in examples:
    print(f"Input : {example}")
    print(f"Output: {respectful_rewrite(example)}")
    print()


## 16. Full Pipeline: Detect, and Transform Only If Needed

This ties Sections 14 and 15 together the way the spec asks: if a comment is
predicted as cyberbullying, we generate a respectful alternative; if not, we keep
the original comment unchanged.

In [ ]:
def moderate_comment(comment):
    result = predict_comment(comment, verbose=False)
    if result['label'] == 1:
        result['final_comment'] = respectful_rewrite(comment)
        result['transformed'] = True
    else:
        result['final_comment'] = comment
        result['transformed'] = False
    return result


for example in ["You are useless and nobody likes you.", "I really liked your presentation."]:
    r = moderate_comment(example)
    print(f"Original Comment: {r['comment']}")
    print(f"Prediction      : {r['label_text']}")
    if r['transformed']:
        print(f"Suggested Respectful Comment: {r['final_comment']}")
    else:
        print("Comment is already classified as non-cyberbullying.")
    print('-' * 80)

## 17. Interactive User Interface (ipywidgets)

A simple in-notebook UI: type a comment, click **Check Comment**, and see the
prediction plus (if needed) the suggested respectful rewrite.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Textarea(
    placeholder='Type a comment here...',
    layout=widgets.Layout(width='600px', height='80px')
)
check_button = widgets.Button(description='Check Comment', button_style='primary')
output_area = widgets.Output()


def on_check_clicked(b):
    with output_area:
        clear_output()
        comment = text_input.value.strip()
        if not comment:
            print("Please enter a comment.")
            return
        r = moderate_comment(comment)
        print(f"Original Comment:\n{r['comment']}\n")
        print(f"Prediction: {r['label_text']}")
        if r['confidence'] is not None:
            print(f"Confidence: {r['confidence']:.2%}")
        print()
        if r['transformed']:
            print(f"Suggested Respectful Comment:\n{r['final_comment']}")
        else:
            print("Comment is already classified as non-cyberbullying.")


check_button.on_click(on_check_clicked)
display(widgets.VBox([text_input, check_button, output_area]))

*(If `ipywidgets` doesn't render in your environment, use this plain-`input()`
fallback instead.)*

In [ ]:
def run_console_ui():
    comment = input("Enter a comment: ")
    r = moderate_comment(comment)
    print(f"\nOriginal Comment: {r['comment']}")
    print(f"Prediction: {r['label_text']}")
    if r['transformed']:
        print(f"Suggested Respectful Comment: {r['final_comment']}")
    else:
        print("Comment is already classified as non-cyberbullying.")

# run_console_ui()   # uncomment to try interactively

## 18. Save the Trained System

We persist the TF-IDF vectorizer and the final chosen model with `joblib`, so the
system can be reloaded later without retraining.

In [ ]:
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
joblib.dump(final_model, 'models/cyberbullying_model.pkl')

# Also save which model was chosen + the preprocessing function's config,
# so a fresh session knows what it's loading.
joblib.dump({'final_model_name': final_model_name}, 'models/meta.pkl')

print("Saved:")
print(" - models/tfidf_vectorizer.pkl")
print(" - models/cyberbullying_model.pkl")
print(" - models/meta.pkl")

### Reload and predict with the saved system

This simulates a fresh session: load the saved vectorizer + model from disk and run
a prediction, with no access to the training variables above.

In [ ]:
loaded_vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
loaded_model = joblib.load('models/cyberbullying_model.pkl')
loaded_meta = joblib.load('models/meta.pkl')

print("Loaded model:", loaded_meta['final_model_name'])

def predict_with_loaded_system(comment):
    cleaned = clean_text(comment)              # same clean_text() function defined in Section 7
    vec = loaded_vectorizer.transform([cleaned])
    pred = loaded_model.predict(vec)[0]
    label_text = 'Cyberbullying' if pred == 1 else 'Not Cyberbullying'
    print(f"Comment: {comment}\nPrediction: {label_text}")
    return label_text

predict_with_loaded_system("Get out, nobody wants you around here.")

## 19. Recommended Project Folder Structure

Organize your deliverable like this before zipping/submitting:

```
cyberbullying_project/
│
├── data/
│   └── cyberbullying_combined.csv
│
├── models/
│   ├── tfidf_vectorizer.pkl
│   └── cyberbullying_model.pkl
│
├── notebooks/
│   └── Cyberbullying_Project.ipynb
│
├── results/
│   ├── graphs/            # all PNGs saved by this notebook
│   └── model_results.csv  # accuracy/precision/recall/F1 table
│
└── README.md
```

In Colab, after running everything, copy the generated files into this structure
(e.g. `!mkdir -p cyberbullying_project/data && !cp data/cyberbullying_combined.csv cyberbullying_project/data/`),
then zip and download it as your submission.

In [ ]:
# Optional: assemble the folder structure automatically at the end of the run
import shutil

proj = 'cyberbullying_project'
os.makedirs(f'{proj}/data', exist_ok=True)
os.makedirs(f'{proj}/models', exist_ok=True)
os.makedirs(f'{proj}/notebooks', exist_ok=True)
os.makedirs(f'{proj}/results/graphs', exist_ok=True)

shutil.copy('data/cyberbullying_combined.csv', f'{proj}/data/')
shutil.copy('models/tfidf_vectorizer.pkl', f'{proj}/models/')
shutil.copy('models/cyberbullying_model.pkl', f'{proj}/models/')
shutil.copy('results/model_results.csv', f'{proj}/results/')
for f in os.listdir('results/graphs'):
    shutil.copy(f'results/graphs/{f}', f'{proj}/results/graphs/')

with open(f'{proj}/README.md', 'w') as f:
    f.write(
        "# Cyberbullying Detection on Social Media Using NLP and Random Forest\n\n"
        "See notebooks/Cyberbullying_Project.ipynb for the full pipeline: dataset "
        "combination, EDA, preprocessing, TF-IDF, model comparison "
        "(Logistic Regression / Naive Bayes / Linear SVM / Random Forest), "
        "the predict_comment() function, and the rule-based respectful-rewrite module.\n"
    )

print(f"Project folder assembled at ./{proj}/")
print("Remember to also copy this notebook file itself into notebooks/Cyberbullying_Project.ipynb")

## End of Pipeline Notebook

The report content, PPT slide topics, technology explanations, and viva Q&A are
provided as a **separate document** (`Cyberbullying_Project_Report_and_Viva_Prep.md`)
so this notebook stays focused on runnable code.